# SA-DMAE Reconstruction Visualization

Pre-trained SA-DMAE의 복원 품질을 시각적으로 확인하는 노트북.

**각 샘플별 출력:**
- **Original** : 원본 center slice (T1ce / T2 / FLAIR)
- **Masked** : 75% 마스킹된 입력 (회색 = 마스킹 영역)
- **Reconstructed** : SA-DMAE 복원 결과
- **Error map** : 복원 오차 (밝을수록 오차 큼)


In [ ]:
# ── Cell 1: 환경 설정 ────────────────────────────────────────────────────────
import os, sys
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/SA-DMAE'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/whkim4338/SA-DMAE.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!pip install timm -q

import torch
print('CUDA:', torch.cuda.is_available())
print('준비 완료')

In [ ]:
# ── Cell 2: 경로 설정 (여기만 수정) ─────────────────────────────────────────

CHECKPOINT = '/content/drive/MyDrive/SA_DMAE_output/checkpoint-best.pth'  # ← 체크포인트 경로
BRATS_PT   = '/content/drive/MyDrive/SA_DMAE_slices/brats'                # ← BraTS .pt 경로
UCSF_PT    = '/content/drive/MyDrive/SA_DMAE_slices/ucsf'                 # ← UCSF .pt 경로
SAVE_DIR   = '/content/drive/MyDrive/SA_DMAE_output/figures'              # ← 결과 저장 경로

N_BRATS    = 4   # BraTS 샘플 수
N_UCSF     = 4   # UCSF 샘플 수
MASK_RATIO = 0.75
SEED       = 42

import os
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'저장 경로: {SAVE_DIR}')

In [ ]:
# ── Cell 3: 모델 로드 ────────────────────────────────────────────────────────
import torch
import sys
sys.path.insert(0, '/content/SA-DMAE')
import models_sa_dmae

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 모델 생성 (학습 때와 동일한 설정)
model = models_sa_dmae.sa_dmae_vit_base_patch16(
    sigma=0.25, n_slices=3, axial_depth=4
)

# 체크포인트 로드
ckpt = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model'])
model.to(device).eval()

print(f'체크포인트 로드 완료: epoch {ckpt["epoch"]}')
print(f'파라미터 수: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')

In [ ]:
# ── Cell 4: 복원 유틸 함수 ───────────────────────────────────────────────────
import torch
import numpy as np

# ImageNet 정규화 상수 (모델과 동일)
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1)


def make_masked_image(original, mask, patch_size=16, mask_value=0.5):
    """
    original : (C, H, W) tensor, 값 범위 [0, 1]
    mask     : (L,) tensor, 1=마스킹된 패치
    반환     : (C, H, W) 마스킹된 이미지
    """
    C, H, W = original.shape
    n_patches = H // patch_size
    masked = original.clone()
    mask_2d = mask.reshape(n_patches, n_patches)  # (14, 14)
    for i in range(n_patches):
        for j in range(n_patches):
            if mask_2d[i, j] == 1:
                masked[:, i*patch_size:(i+1)*patch_size,
                           j*patch_size:(j+1)*patch_size] = mask_value
    return masked


def reconstruct(model, sample, mask_ratio, device, seed=None):
    """
    sample : (3, 3, 224, 224) tensor (n_slices, C, H, W)
    반환   : original, masked_img, recon — 모두 (C, H, W), 값 [0, 1]
    """
    if seed is not None:
        torch.manual_seed(seed)

    x = sample.unsqueeze(0).to(device)  # (1, 3, 3, 224, 224)

    with torch.no_grad():
        loss, pred, mask = model(x, mask_ratio=mask_ratio)

    # pred: (1, 196, 16*16*3), 정규화 공간 → 픽셀 공간으로 denormalize
    recon_patches = pred[0].cpu()  # (196, 768)
    recon_img = model.unpatchify(pred)[0].cpu()  # (C, H, W), 정규화 공간

    # denormalize: x_pixel = x_norm * std + mean
    mean = IMAGENET_MEAN.squeeze(0)  # (3, 1, 1)
    std  = IMAGENET_STD.squeeze(0)
    recon_img = (recon_img * std + mean).clamp(0, 1)  # (C, H, W)

    center_idx  = sample.shape[0] // 2
    original    = sample[center_idx].cpu()   # (C, H, W), [0,1]
    mask_1d     = mask[0].cpu()              # (196,)
    masked_img  = make_masked_image(original, mask_1d)

    return original, masked_img, recon_img, mask_1d, loss.item()


print('유틸 함수 준비 완료')

In [ ]:
# ── Cell 5: 시각화 함수 ──────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

MOD_NAMES = ['T1ce', 'T2', 'FLAIR']
COL_LABELS = ['Original', 'Masked (75%)', 'Reconstructed', 'Error Map']


def visualize_sample(original, masked, recon, case_id, loss_val,
                     save_path=None):
    """
    한 케이스의 3 modality × 4 열 시각화
    행: T1ce / T2 / FLAIR
    열: Original / Masked / Reconstructed / Error Map
    """
    fig, axes = plt.subplots(3, 4, figsize=(14, 11))
    fig.patch.set_facecolor('#111111')
    fig.suptitle(
        f'{case_id}  —  MSE Loss: {loss_val:.4f}',
        fontsize=13, fontweight='bold', color='white', y=0.98
    )

    # 열 제목
    for col, label in enumerate(COL_LABELS):
        axes[0, col].set_title(label, fontsize=11, color='#AAAAAA', pad=6)

    for row, mod in enumerate(MOD_NAMES):
        orig_ch  = original[row].numpy()
        mask_ch  = masked[row].numpy()
        recon_ch = recon[row].numpy()
        error    = np.abs(orig_ch - recon_ch)

        imgs   = [orig_ch, mask_ch, recon_ch, error]
        cmaps  = ['gray', 'gray', 'gray', 'hot']
        vmins  = [0, 0, 0, 0]
        vmaxs  = [1, 1, 1, error.max() + 1e-6]

        for col, (img, cmap, vmin, vmax) in enumerate(zip(imgs, cmaps, vmins, vmaxs)):
            ax = axes[row, col]
            ax.set_facecolor('black')
            im = ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
            ax.axis('off')

            # 행 레이블 (맨 왼쪽)
            if col == 0:
                ax.set_ylabel(mod, fontsize=11, color='white',
                              fontweight='bold', labelpad=6)
                ax.yaxis.set_label_position('left')
                ax.tick_params(left=False, labelleft=False)

            # Error map: PSNR 표시
            if col == 3:
                mse  = np.mean((orig_ch - recon_ch) ** 2)
                psnr = 10 * np.log10(1.0 / (mse + 1e-8))
                ax.text(0.03, 0.04, f'PSNR: {psnr:.1f} dB',
                        transform=ax.transAxes, fontsize=8.5,
                        color='white', va='bottom',
                        bbox=dict(fc='black', alpha=0.6, pad=2))

            # Reconstructed: 픽셀 범위 표시
            if col == 2:
                ax.text(0.03, 0.04, f'[{recon_ch.min():.2f}, {recon_ch.max():.2f}]',
                        transform=ax.transAxes, fontsize=8,
                        color='lightgray', va='bottom',
                        bbox=dict(fc='black', alpha=0.5, pad=2))

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    if save_path:
        fig.savefig(save_path, dpi=130, bbox_inches='tight',
                    facecolor=fig.get_facecolor())
        print(f'  저장: {save_path}')
    plt.show()
    plt.close()


print('시각화 함수 준비 완료')

In [ ]:
# ── Cell 6: BraTS 샘플 복원 시각화 ──────────────────────────────────────────
import random
from pathlib import Path

torch.manual_seed(SEED)
random.seed(SEED)

brats_files = sorted(Path(BRATS_PT).glob('*.pt'))
brats_samples = random.sample(brats_files, min(N_BRATS, len(brats_files)))

print(f'=== BraTS 2021 ({N_BRATS}개 샘플) ===')
brats_results = []

for pt_path in brats_samples:
    sample   = torch.load(pt_path, map_location='cpu', weights_only=True)
    case_id  = pt_path.stem
    orig, masked, recon, mask, loss_val = reconstruct(
        model, sample, MASK_RATIO, device, seed=SEED
    )
    brats_results.append((case_id, orig, masked, recon, loss_val))

    save_path = f'{SAVE_DIR}/brats_{case_id}.png'
    print(f'\n[BraTS] {case_id}  loss={loss_val:.4f}')
    visualize_sample(orig, masked, recon, f'[BraTS] {case_id}',
                     loss_val, save_path)

In [ ]:
# ── Cell 7: UCSF 샘플 복원 시각화 ───────────────────────────────────────────
ucsf_files   = sorted(Path(UCSF_PT).glob('*.pt'))
ucsf_samples = random.sample(ucsf_files, min(N_UCSF, len(ucsf_files)))

print(f'=== UCSF-PDGM ({N_UCSF}개 샘플) ===')
ucsf_results = []

for pt_path in ucsf_samples:
    sample  = torch.load(pt_path, map_location='cpu', weights_only=True)
    case_id = pt_path.stem
    orig, masked, recon, mask, loss_val = reconstruct(
        model, sample, MASK_RATIO, device, seed=SEED
    )
    ucsf_results.append((case_id, orig, masked, recon, loss_val))

    save_path = f'{SAVE_DIR}/ucsf_{case_id}.png'
    print(f'\n[UCSF] {case_id}  loss={loss_val:.4f}')
    visualize_sample(orig, masked, recon, f'[UCSF] {case_id}',
                     loss_val, save_path)

In [ ]:
# ── Cell 8: 논문용 그리드 Figure (BraTS + UCSF 각 2개씩) ─────────────────────
# 행: BraTS2개 + UCSF2개 = 4개 케이스
# 열: Original T1ce / Masked T1ce / Recon T1ce / Original FLAIR / Masked FLAIR / Recon FLAIR
import numpy as np

selected = brats_results[:2] + ucsf_results[:2]
row_labels = [
    '[BraTS]', '[BraTS]',
    '[UCSF]',  '[UCSF]',
]

fig, axes = plt.subplots(4, 8, figsize=(22, 12))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle('SA-DMAE Reconstruction Results  (mask ratio=75%)',
             fontsize=15, fontweight='bold', color='white', y=1.01)

col_titles = [
    'Original\nT1ce', 'Masked\nT1ce', 'Recon\nT1ce', 'Error\nT1ce',
    'Original\nFLAIR', 'Masked\nFLAIR', 'Recon\nFLAIR', 'Error\nFLAIR',
]
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=9, color='#BBBBBB',
                           fontweight='bold', pad=4)

for row, (case_id, orig, masked, recon, loss_val) in enumerate(selected):
    label = row_labels[row]

    # T1ce (modality 0), FLAIR (modality 2)
    data = [
        orig[0].numpy(), masked[0].numpy(), recon[0].numpy(),  # T1ce
        orig[2].numpy(), masked[2].numpy(), recon[2].numpy(),  # FLAIR
    ]
    errors = [
        np.abs(orig[0].numpy() - recon[0].numpy()),
        np.abs(orig[2].numpy() - recon[2].numpy()),
    ]
    all_imgs  = data[:3] + [errors[0]] + data[3:] + [errors[1]]
    all_cmaps = ['gray']*3 + ['hot'] + ['gray']*3 + ['hot']

    for col, (img, cmap) in enumerate(zip(all_imgs, all_cmaps)):
        ax = axes[row, col]
        ax.set_facecolor('black')
        vmax = img.max() + 1e-6 if cmap == 'hot' else 1.0
        ax.imshow(img, cmap=cmap, vmin=0, vmax=vmax)
        ax.axis('off')

    # 행 레이블
    axes[row, 0].set_ylabel(
        f'{label}\n{case_id[:20]}\nloss={loss_val:.4f}',
        fontsize=7.5, color='white', labelpad=4,
        rotation=0, ha='right', va='center',
    )

plt.tight_layout()
grid_path = f'{SAVE_DIR}/paper_figure_reconstruction.png'
fig.savefig(grid_path, dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print(f'논문용 Figure 저장: {grid_path}')

In [ ]:
# ── Cell 9: 정량적 결과 요약 ─────────────────────────────────────────────────
import numpy as np

def compute_metrics(original, recon):
    """채널별 평균 PSNR / MSE 계산."""
    mse_vals, psnr_vals = [], []
    for c in range(original.shape[0]):
        o = original[c].numpy()
        r = recon[c].numpy()
        mse  = np.mean((o - r) ** 2)
        psnr = 10 * np.log10(1.0 / (mse + 1e-8))
        mse_vals.append(mse)
        psnr_vals.append(psnr)
    return np.mean(mse_vals), np.mean(psnr_vals)

print('=' * 60)
print(f'{"Case ID":<35} {"MSE":>8}  {"PSNR (dB)":>10}')
print('-' * 60)

all_mse, all_psnr = [], []

for case_id, orig, masked, recon, loss_val in brats_results:
    mse, psnr = compute_metrics(orig, recon)
    all_mse.append(mse); all_psnr.append(psnr)
    print(f'[BraTS] {case_id:<28} {mse:>8.4f}  {psnr:>10.2f}')

for case_id, orig, masked, recon, loss_val in ucsf_results:
    mse, psnr = compute_metrics(orig, recon)
    all_mse.append(mse); all_psnr.append(psnr)
    print(f'[UCSF]  {case_id:<28} {mse:>8.4f}  {psnr:>10.2f}')

print('=' * 60)
print(f'{"평균":<35} {np.mean(all_mse):>8.4f}  {np.mean(all_psnr):>10.2f}')
print(f'{"표준편차":<35} {np.std(all_mse):>8.4f}  {np.std(all_psnr):>10.2f}')